# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example for loading and exploring a dataset using the `mlcroissant` library following [the Croissant FAIR data protocol](https://mlcommons.org/croissant/). In this notebook, we use the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json), which includes demographic details and assessments such as GAD-7, PHQ-9, and PSQ.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata and create the Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their Croissant `@id` fields.

In [ ]:
# List all record sets in the dataset, print their @id and properties
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '')}, description: {record_set.get('description', '')}")
    print("  Fields:")
    for field in record_set.get('fields', []):
        fid = field.get('@id', field)
        # If field is expanded, print name, else just @id
        if isinstance(field, dict):
            print(f"    - @id: {fid}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
        else:
            print(f"    - @id: {fid}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We demonstrate loading the main survey table. **Always use the record set and field `@id`s shown above.**

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print(f"Record set IDs: {record_set_ids}")

# Load all record sets into DataFrames for exploration
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# For demonstration, let's pick the main record set as the first one (often survey data)
main_record_set_id = record_set_ids[0]
print(f"\nMain Record Set (@id): {main_record_set_id}")
print("Columns:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This demonstrates further processing for ML/AI readiness.

In [ ]:
# Select a numeric field for demonstration - e.g., 'gad7_total_score' or PHQ-9/PSQ total
# For the FAIR² dataset, example field IDs might be:
# - 'gad7_total_score'
# - 'phq9_total_score'
# - 'psq_total_score'
# Substitute with the actual @id, e.g. 'gad7_total_score'

numeric_field = 'gad7_total_score'  # Use the actual field @id if different
main_df = dataframes[main_record_set_id]

# For demonstration, filter participants with GAD-7 total score > 10 (suggesting moderate anxiety)
threshold = 10
if numeric_field in main_df.columns:
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field, e.g., 'village' or 'gender', using @id
    group_field = 'village'  # Change this to the actual field @id as needed
    if group_field in main_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} by {group_field} for filtered records:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field}' not found in columns: {main_df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the GAD-7 total score (replace if using another field)
if numeric_field in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print(f"No field '{numeric_field}' found for visualization.")

## 6. Conclusion
This notebook demonstrated loading the FAIR² survey dataset using `mlcroissant`, inspecting all record sets and their fields by `@id`, extracting data into DataFrames, performing basic EDA (filtering, normalization, grouping), and visualizing key distributions. 

You may further extend these analyses by referencing additional field `@id`s from the record set overview, and combining with other data sources facilitated by the Croissant metadata schema.